<a href="https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**  
This notebook builds the full feature vector (engineered features, missing handling, categorical encoding), documents each feature’s availability timeline, hunts every form of leakage, and lists excluded columns with reasons.

In [1]:
%pip install -q duckdb scikit-learn pandas numpy
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

# Paths – same as ML-04, but we may load more months for feature engineering if needed
FACT_MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

## 1. Build the feature vector
One row per content item × client, with 8 engineered features.  
All features are computed from March 2026 data and static content metadata — nothing peeks into the future.

In [2]:
# 1. Monthly aggregates from the fact table
monthly = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_march,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_march,
        MAX(CASE WHEN gsc_impressions > 0 THEN report_date END) AS last_seen_date
    FROM {FACT_MONTH}
    GROUP BY client_hash_id, content_hash_id
""").df()

# 2. Content metadata (static attributes)
content = con.sql(f"""
    SELECT
        content_hash_id,
        word_count,
        content_type,
        content_created_date,
        is_published
    FROM {DIM_CONTENT}
""").df()

# 3. Join and create engineered features
df = monthly.merge(content, on='content_hash_id', how='left')

# Feature engineering
df['ctr_march'] = df['clicks_march'] / df['imp_march'].replace(0, None) * 100
df['engagement_rate_march'] = df['engaged_sessions_march'] / df['sessions_march'].replace(0, None) * 100
df['days_since_created'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days
df['is_evergreen'] = (df['content_type'] == 'evergreen').astype(int)  # categorical encoded as binary
# df['content_length_bin'] = pd.cut(df['word_count'], bins=[0, 300, 800, 2000, np.inf], labels=['short','medium','long','very_long'])

# Fill missing values
df['engagement_rate_march'] = df['engagement_rate_march'].fillna(0)          # no GA4 → 0 engagement
df['days_since_created'] = df['days_since_created'].fillna(df['days_since_created'].median())
df['ctr_march'] = df['ctr_march'].fillna(0)

# Drop rows with no impressions (no signal)
df = df[df['imp_march'] > 0].copy()

# Final feature set
feature_cols = [
    'imp_march', 'clicks_march', 'avg_position_march', 'ctr_march',
    'engagement_rate_march', 'days_since_created', 'is_evergreen', 'word_count'
]

X = df[feature_cols].copy()
print(f"Feature vector shape: {X.shape}")
X.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_3094/654487352.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['engagement_rate_march'] = df['engagement_rate_march'].fillna(0)          # no GA4 → 0 engagement
/tmp/ipykernel_3094/654487352.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ctr_march'] = df['ctr_march'].fillna(0)


Feature vector shape: (176738, 8)


,imp_march,clicks_march,avg_position_march,ctr_march,engagement_rate_march,days_since_created,is_evergreen,word_count
0,77.0,0.0,4.888929,0.000000,0.0,47,0,3579
1,602.0,4.0,4.428747,0.664452,0.0,47,0,2455
2,810.0,1.0,4.866123,0.123457,0.0,47,0,3653
3,82.0,0.0,10.100347,0.000000,0.0,47,0,3096
4,1858.0,6.0,1.854929,0.322928,0.0,47,0,3305


## 2. Feature notes (meaning, missing, categorical, available‑when?)
Every feature is described below.

In [4]:
feature_notes = {
    'imp_march': {
        'meaning': 'Total GSC impressions in March 2026',
        'missing': 'None (filtered rows with 0 impressions)',
        'categorical': False,
        'available_when': 'End of March – fully observed from daily logs'
    },
    'clicks_march': {
        'meaning': 'Total GSC clicks in March',
        'missing': 'None',
        'categorical': False,
        'available_when': 'End of March'
    },
    'avg_position_march': {
        'meaning': 'Average GSC position across March (ignoring 0‑position rows)',
        'missing': 'None (NULLIF avoids 0)',
        'categorical': False,
        'available_when': 'End of March'
    },
    'ctr_march': {
        'meaning': 'Click‑through rate (%) in March',
        'missing': 'Filled with 0 where impressions = 0',
        'categorical': False,
        'available_when': 'End of March'
    },
    'engagement_rate_march': {
        'meaning': 'GA4 engaged sessions / sessions (%), filtered to clients with GA4 data',
        'missing': 'Set to 0 for rows without GA4',
        'categorical': False,
        'available_when': 'End of March, but only for clients who have GA4 tracking live'
    },
    'days_since_created': {
        'meaning': 'Number of days from content creation to 2026‑03‑31',
        'missing': 'Median‑imputed if creation date missing',
        'categorical': False,
        'available_when': 'The moment the content was created (static attribute)'
    },
    'is_evergreen': {
        'meaning': '1 if content_type = "evergreen", else 0',
        'missing': 'None (content_type always present)',
        'categorical': True,
        'available_when': 'At content creation time'
    },
    'word_count': {
        'meaning': 'Number of words in the content',
        'missing': 'None (from dim_content)',
        'categorical': False,
        'available_when': 'Content creation time'
    }
}

pd.DataFrame(feature_notes).T

,meaning,missing,categorical,available_when
imp_march,Total GSC impressions in March 2026,None (filtered rows with 0 impressions),False,End of March – fully observed from daily logs
clicks_march,Total GSC clicks in March,None,False,End of March
avg_position_march,Average GSC position across March (ignoring 0‑...,None (NULLIF avoids 0),False,End of March
ctr_march,Click‑through rate (%) in March,Filled with 0 where impressions = 0,False,End of March
engagement_rate_march,"GA4 engaged sessions / sessions (%), filtered ...",Set to 0 for rows without GA4,False,"End of March, but only for clients who have GA..."
days_since_created,Number of days from content creation to 2026‑0...,Median‑imputed if creation date missing,False,The moment the content was created (static att...
is_evergreen,"1 if content_type = ""evergreen"", else 0",None (content_type always present),True,At content creation time
word_count,Number of words in the content,None (from dim_content),False,Content creation time


## 3. The leakage hunt
Three deliberate attacks:
1. A column that directly encodes the label (future decline) inside the features – the same trap from ML‑04, but now in the full feature vector.
2. A feature that uses data from a future window (e.g., a rolling average that extends into April).
3. A product‑flag‑style column (e.g., a content's “priority” score derived from the label itself).

Every attack is shown, measured, and then removed.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Create the proxy label – decline in second half vs first half of March (same as ML‑04)
label_src = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_last15,
        SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_prev15
    FROM {FACT_MONTH}
    GROUP BY content_hash_id, client_hash_id
""").df()

df = df.merge(label_src, on=['content_hash_id','client_hash_id'], how='left')
df['is_declining'] = (
    (df['impr_prev15'] > 0) & ((df['impr_last15'] - df['impr_prev15']) / df['impr_prev15'] < -0.20)
).astype(int)

# Base honest features
honest_features = ['imp_march','clicks_march','avg_position_march','ctr_march',
                   'engagement_rate_march','days_since_created','is_evergreen','word_count']
X_honest = df[honest_features].fillna(0)
y = df['is_declining']

# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)

# ---- Attack 1: label itself as feature (impr_last15 - impr_prev15) ----
X_tr_leak1 = X_tr.copy()
X_tr_leak1['future_impression_change'] = df.loc[X_tr.index, 'impr_last15'] - df.loc[X_tr.index, 'impr_prev15']
X_te_leak1 = X_te.copy()
X_te_leak1['future_impression_change'] = df.loc[X_te.index, 'impr_last15'] - df.loc[X_te.index, 'impr_prev15']

# ---- Attack 2: feature from a future window (April 2026) ----
# Simulate: we pull a "trend_april" from the label itself (for demonstration, since actual April data isn't loaded)
# In real scenario, this would be a column derived from April data. Here we just use the label again.
X_tr_leak2 = X_tr.copy()
X_tr_leak2['trend_april'] = y_tr  # This would be a number derived from April, but we cheat directly
X_te_leak2 = X_te.copy()
X_te_leak2['trend_april'] = y_te

# ---- Attack 3: product score that encodes label (priority_flag = label) ----
X_tr_leak3 = X_tr.copy()
X_tr_leak3['priority_flag'] = y_tr
X_te_leak3 = X_te.copy()
X_te_leak3['priority_flag'] = y_te

# Evaluate honest model
model_honest = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
acc_honest = accuracy_score(y_te, model_honest.predict(X_te))
print(f"Honest accuracy (8 features): {acc_honest:.4f}")

# Evaluate Attack 1
model_leak1 = LogisticRegression(max_iter=1000).fit(X_tr_leak1, y_tr)
acc_leak1 = accuracy_score(y_te, model_leak1.predict(X_te_leak1))
print(f"Leak Attack 1 accuracy (label as feature): {acc_leak1:.4f}")

# Evaluate Attack 2
model_leak2 = LogisticRegression(max_iter=1000).fit(X_tr_leak2, y_tr)
acc_leak2 = accuracy_score(y_te, model_leak2.predict(X_te_leak2))
print(f"Leak Attack 2 accuracy (future window): {acc_leak2:.4f}")

# Evaluate Attack 3
model_leak3 = LogisticRegression(max_iter=1000).fit(X_tr_leak3, y_tr)
acc_leak3 = accuracy_score(y_te, model_leak3.predict(X_te_leak3))
print(f"Leak Attack 3 accuracy (product flag = label): {acc_leak3:.4f}")

# Show the jump
print(f"\nHonest → Leak jump: +{acc_leak1 - acc_honest:.4f} (Attack 1), +{acc_leak2 - acc_honest:.4f} (Attack 2), +{acc_leak3 - acc_honest:.4f} (Attack 3)")
print("All leak columns deleted – final feature set remains the 8 honest ones.")

Honest accuracy (8 features): 0.7189
Leak Attack 1 accuracy (label as feature): 1.0000
Leak Attack 2 accuracy (future window): 1.0000
Leak Attack 3 accuracy (product flag = label): 1.0000

Honest → Leak jump: +0.2811 (Attack 1), +0.2811 (Attack 2), +0.2811 (Attack 3)
All leak columns deleted – final feature set remains the 8 honest ones.


## 4. What I excluded and why
A list of fields I deliberately refused to use, each with a one‑line reason.

In [6]:
excluded = pd.DataFrame({
    'field': [
        'ga4_sessions, ga4_users, ga4_pageviews (unfiltered)',
        'gsc_sum_position',
        'channel breakdowns (sessions_organic, sessions_direct, ...)',
        'AI traffic columns (ai_chatgpt, ai_perplexity, ...)',
        'scroll_events',
        'client_hash_id, content_hash_id, url_hash_id, keyword_hash_id',
        'content_updated_date, last_optimized_date',
        'is_published',
        'keyword_char_count, keyword_token_count',
        'url_char_count',
        'backlinks, search_volume, competition, cpc',
        'provider_used, model_used'
    ],
    'reason': [
        'Zero means “no tracking”, not “no traffic” – would teach false pattern.',
        'Redundant with avg_position; not an independent signal.',
        'Sparse and client‑specific; would overfit to individual client patterns.',
        'AI traffic still in early adoption – noisy and client‑specific.',
        'Collection quality varies wildly across clients; unreliable.',
        'Identifiers – would allow memorisation, not learning.',
        'Could encode future editorial decisions (leakage if optimisation happened after March).',
        'Almost all rows are True – near‑zero variance in our filtered set.',
        'Not meaningful for content ranking; noise.',
        'Not a signal for performance.',
        'External metrics not updated daily; stale or missing for many rows.',
        'Metadata about how content was generated, not about its performance.'
    ]
})

excluded

,field,reason
0,"ga4_sessions, ga4_users, ga4_pageviews (unfilt...","Zero means “no tracking”, not “no traffic” – w..."
1,gsc_sum_position,Redundant with avg_position; not an independen...
2,"channel breakdowns (sessions_organic, sessions...",Sparse and client‑specific; would overfit to i...
3,"AI traffic columns (ai_chatgpt, ai_perplexity,...",AI traffic still in early adoption – noisy and...
4,scroll_events,Collection quality varies wildly across client...
5,"client_hash_id, content_hash_id, url_hash_id, ...","Identifiers – would allow memorisation, not le..."
6,"content_updated_date, last_optimized_date",Could encode future editorial decisions (leaka...
7,is_published,Almost all rows are True – near‑zero variance ...
8,"keyword_char_count, keyword_token_count",Not meaningful for content ranking; noise.
9,url_char_count,Not a signal for performance.


## Self-check
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.